In [39]:
import pandas as pd

In [40]:
aqi_df = pd.read_csv("../data/processed/clean_aqi_pm25.csv")
weather_df = pd.read_csv("../data/raw/openmeteo_delhi_sample.csv")

In [41]:

weather_df.shape

(1440, 7)

In [42]:
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime_local        1440 non-null   str    
 1   temperature_2m        1440 non-null   float64
 2   relative_humidity_2m  1440 non-null   int64  
 3   wind_speed_10m        1440 non-null   float64
 4   wind_direction_10m    1440 non-null   int64  
 5   precipitation         1440 non-null   float64
 6   hour                  1440 non-null   str    
dtypes: float64(3), int64(2), str(2)
memory usage: 78.9 KB


In [43]:
weather_df.columns

Index(['datetime_local', 'temperature_2m', 'relative_humidity_2m',
       'wind_speed_10m', 'wind_direction_10m', 'precipitation', 'hour'],
      dtype='str')

In [44]:
weather_df["datetime_local"].head()

0    2024-12-23 00:00:00
1    2024-12-23 01:00:00
2    2024-12-23 02:00:00
3    2024-12-23 03:00:00
4    2024-12-23 04:00:00
Name: datetime_local, dtype: str

In [45]:
aqi_df["datetime_local"] = pd.to_datetime(aqi_df["datetime_local"])
weather_df["datetime_local"] = pd.to_datetime(weather_df["datetime_local"])

In [46]:
aqi_df["datetime_local"].head()


0   2025-02-19 03:30:00+05:30
1   2025-02-19 03:45:00+05:30
2   2025-02-19 04:00:00+05:30
3   2025-02-19 04:15:00+05:30
4   2025-02-19 04:30:00+05:30
Name: datetime_local, dtype: datetime64[us, UTC+05:30]

In [47]:
aqi_df["datetime_local"] = aqi_df["datetime_local"].dt.tz_localize(None)

In [48]:
aqi_df.info()
weather_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   location_id     252 non-null    int64         
 1   location_name   252 non-null    str           
 2   sensor_id       252 non-null    int64         
 3   parameter       252 non-null    str           
 4   value           252 non-null    float64       
 5   unit            252 non-null    str           
 6   datetime_utc    252 non-null    str           
 7   datetime_local  252 non-null    datetime64[us]
 8   latitude        252 non-null    float64       
 9   longitude       252 non-null    float64       
 10  hour            252 non-null    str           
dtypes: datetime64[us](1), float64(3), int64(2), str(5)
memory usage: 21.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         

In [49]:
aqi_df["hour"] = aqi_df["datetime_local"].dt.floor("h")
weather_df["hour"] = weather_df["datetime_local"].dt.floor("h")

In [50]:
aqi_df[["datetime_local", "hour"]].head()
    # because aqi wasnt hourly unlike weather, thus change it and check if it is now hourly
weather_df[["datetime_local", "hour"]].head()

,datetime_local,hour
0,2024-12-23 00:00:00,2024-12-23 00:00:00
1,2024-12-23 01:00:00,2024-12-23 01:00:00
2,2024-12-23 02:00:00,2024-12-23 02:00:00
3,2024-12-23 03:00:00,2024-12-23 03:00:00
4,2024-12-23 04:00:00,2024-12-23 04:00:00


In [51]:
#check if weather columns have any missing values
weather_df.isna().sum()

datetime_local          0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
hour                    0
dtype: int64

In [52]:
weather_clean_df = weather_df[
    [
        "datetime_local",
        "hour",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_direction_10m",
        "precipitation"
    ]
].copy()
#hour is where the merging will happen, thus datetime_local is not needed

In [53]:
weather_clean_df.to_csv("../data/processed/clean_weather.csv", index=False)

In [54]:
#merging weatherto the aqi readings
merged_df = pd.merge(
    aqi_df,
    weather_clean_df,
    on="hour",
    how="left"
)

merged_df.head()

,location_id,location_name,sensor_id,parameter,value,unit,datetime_utc,datetime_local_x,latitude,longitude,hour,datetime_local_y,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation
0,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,105.0,µg/m³,2025-02-18 22:00:00+00:00,2025-02-19 03:30:00,28.646835,77.316032,2025-02-19 03:00:00,2025-02-19 03:00:00,15.6,68,5.1,280,0.0
1,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:15:00+00:00,2025-02-19 03:45:00,28.646835,77.316032,2025-02-19 03:00:00,2025-02-19 03:00:00,15.6,68,5.1,280,0.0
2,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:30:00+00:00,2025-02-19 04:00:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0
3,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:45:00+00:00,2025-02-19 04:15:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0
4,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 23:00:00+00:00,2025-02-19 04:30:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0


In [55]:
merged_df.shape

(252, 17)

In [56]:
merged_df.isna().sum()

location_id             0
location_name           0
sensor_id               0
parameter               0
value                   0
unit                    0
datetime_utc            0
datetime_local_x        0
latitude                0
longitude               0
hour                    0
datetime_local_y        0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
dtype: int64

In [57]:
aqi_df["hour"].min(), aqi_df["hour"].max()

(Timestamp('2025-02-19 01:00:00'), Timestamp('2025-02-19 23:00:00'))

In [58]:
weather_clean_df["hour"].min(), weather_clean_df["hour"].max()

(Timestamp('2024-12-23 00:00:00'), Timestamp('2025-02-20 23:00:00'))

In [59]:
latest_aqi_date = aqi_df["hour"].max().date()
latest_aqi_date

datetime.date(2025, 2, 19)

## Fix: Rebuild 60-day AQI + historical weather window

In [38]:
latest_aqi_time = aqi_df["hour"].max()
latest_aqi_date = latest_aqi_time.date()

start_date = (latest_aqi_time - pd.Timedelta(days=59)).date()
end_date = latest_aqi_date

start_date, end_date

(datetime.date(2024, 12, 23), datetime.date(2025, 2, 20))

In [61]:
merged_df.isna().sum()

location_id             0
location_name           0
sensor_id               0
parameter               0
value                   0
unit                    0
datetime_utc            0
datetime_local_x        0
latitude                0
longitude               0
hour                    0
datetime_local_y        0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
dtype: int64

In [63]:
merged_df.to_csv("../data/processed/merged_aqi_weather.csv", index=False)

In [64]:
check_merged = pd.read_csv("../data/processed/merged_aqi_weather.csv")
check_merged.head()

,location_id,location_name,sensor_id,parameter,value,unit,datetime_utc,datetime_local_x,latitude,longitude,hour,datetime_local_y,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation
0,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,105.0,µg/m³,2025-02-18 22:00:00+00:00,2025-02-19 03:30:00,28.646835,77.316032,2025-02-19 03:00:00,2025-02-19 03:00:00,15.6,68,5.1,280,0.0
1,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:15:00+00:00,2025-02-19 03:45:00,28.646835,77.316032,2025-02-19 03:00:00,2025-02-19 03:00:00,15.6,68,5.1,280,0.0
2,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:30:00+00:00,2025-02-19 04:00:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0
3,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 22:45:00+00:00,2025-02-19 04:15:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0
4,235,"Anand Vihar, New Delhi - DPCC",12235610,pm25,95.0,µg/m³,2025-02-18 23:00:00+00:00,2025-02-19 04:30:00,28.646835,77.316032,2025-02-19 04:00:00,2025-02-19 04:00:00,15.1,69,5.1,288,0.0


In [65]:
check_merged.shape

(252, 17)